In [ ]:
# 1000QTSA test
import json
from collections import defaultdict

def detailed_accuracy_analysis(jsonl_file_path):
    results = {
        "total_questions": 0,
        "models": {
            "(new)1B_prompt2": {"correct": 0, "incorrect": 0, "details": []},
            "(new)finetuned_1B_prompt2": {"correct": 0, "incorrect": 0, "details": []},
            "(new)1B_Instruct_prompt2": {"correct": 0, "incorrect": 0, "details": []},
            "(new)finetuned_1B_Instruct_prompt2": {"correct": 0, "incorrect": 0, "details": []},
            "(new)3B_prompt2": {"correct": 0, "incorrect": 0, "details": []},
            "(new)finetuned_3B_prompt2": {"correct": 0, "incorrect": 0, "details": []},
            "(new)3B_Instruct_prompt2": {"correct": 0, "incorrect": 0, "details": []},
            "(new)finetuned_3B_Instruct_prompt2": {"correct": 0, "incorrect": 0, "details": []},
        }
    }
    
    with open(jsonl_file_path, 'r', encoding='utf-8') as file:
        for line_num, line in enumerate(file, 1):
            if line.strip():
                try:
                    data = json.loads(line)
                    results["total_questions"] += 1
                    correct_answer = data["Correct Answer"]
                    question_id = data.get("ID", line_num)
                    
                    for model in results["models"].keys():
                        model_answer = data.get(f"{model} Answer", "")
                        if model_answer is None:
                            model_answer = ""

                        if not isinstance(model_answer, str):
                            model_answer = str(model_answer)
                            
                        is_correct = sorted(model_answer) == sorted(correct_answer)
                        
                        if is_correct:
                            results["models"][model]["correct"] += 1
                        else:
                            results["models"][model]["incorrect"] += 1
                     
                        results["models"][model]["details"].append({
                            "question_id": question_id,
                            "correct": is_correct,
                            "model_answer": model_answer,
                            "correct_answer": correct_answer
                        })
                
                except json.JSONDecodeError:
                    print(f"Warning: JSON format error on line {line_num}")
                except KeyError as e:
                    print(f"Warning: Missing required field on line {line_num}: {e}")
 
    for model in results["models"]:
        total = results["models"][model]["correct"] + results["models"][model]["incorrect"]
        if total > 0:
            accuracy = results["models"][model]["correct"] / total
            results["models"][model]["accuracy"] = round(accuracy, 4)
        else:
            results["models"][model]["accuracy"] = 0
    
    return results

def print_detailed_results(results):
    print("=" * 60)
    print(f"Total questions: {results['total_questions']}")
    print()
    
    for model, stats in results["models"].items():
        print(f"{model} model:")
        print(f"  Correct: {stats['correct']}")
        print(f"  Incorrect: {stats['incorrect']}")
        print(f"  Accuracy: {stats['accuracy'] * 100:.2f}%")
        print("-" * 40)

if __name__ == "__main__":
    jsonl_file_path = "Outcome_1000.jsonl" 
    
    try:
        detailed_results = detailed_accuracy_analysis(jsonl_file_path)
        print_detailed_results(detailed_results)
        
    except Exception as e:
        print(f"Error: {e}")